# CheckIt.AI — Rapport d’exploration des sources

**Étape 01 — Explorez et qualifiez les sources de données**  
**Date :** 26 août 2026

## 1. Objectif du rapport

Ce rapport identifie et compare des sources capables de fournir des publications contenant du **texte et une image associés**. Ces données serviront, dans la suite du projet, à entraîner ou évaluer un détecteur de désinformation multimodale.

Dans ce document, le traitement est toutefois limité à la partie **workflow ETL**, tout en gardant en tête l'usage qui sera fait au final des données.

**<u>Workflow ETL envisagé :</u>**

1. extraire les données depuis plusieurs types de sources ;
2. transformer et normaliser les publications ;
3. contrôler la qualité des couples texte-image ;
4. charger les données dans des fichiers réutilisables.


## 2. Critères utilisés pour qualifier une source

Une source est considérée comme intéressante si elle permet de récupérer la majorité des éléments suivants :

| Élément | Utilité dans l’ETL |
|---|---|
| Identifiant de la publication | Éviter les doublons et mettre à jour une entrée |
| Titre et texte | Conserver le contenu éditorial |
| Nom de domaine | Avoir la source de l'information |
| URL de l’image | Télécharger l’image liée à la publication |
| URL de la publication | Assurer la traçabilité |
| Date de publication | Trier et filtrer les données |
| Source et nom de domaine | Identifier l’origine du contenu |
| Auteur | Champ secondaire utile, lorsqu’il existe |
| Langue | Filtrer ou constituer un corpus multilingue |
| Fiabilité fournie | Conserver une éventuelle annotation vrai/faux |
| Origine du label | Savoir qui a produit l’annotation et selon quelle méthode |
| Date de collecte | Suivre les exécutions du pipeline |


### Échelle de qualité des labels

- **Élevée** : verdict produit par une organisation de fact-checking, avec justification et URL de preuve.
- **Moyenne** : label exploitable, mais produit automatiquement, déduit d’une catégorie ou dépendant d’une source moins rigoureuse.
- **Faible** : vote communautaire, flair ou opinion d’un utilisateur sans vérification externe.
- **Absente** : la source fournit des actualités, mais aucun verdict vrai/faux.

Un label décrit la méthode d’annotation d’un jeu de données ; il ne constitue jamais une vérité absolue. L’ETL le conserve avec sa provenance sans le modifier.

## 3. Cas typiques de désinformation multimodale

Les sources recherchées doivent permettre d’observer plusieurs cas :

- **image sortie de son contexte** : l’image est réelle, mais la légende, le lieu ou la date sont faux ;
- **fausse connexion** : le titre ou l’image ne correspond pas au contenu de l’article ;
- **image manipulée ou générée** : des éléments ont été ajoutés, supprimés ou créés ;
- **usurpation de source** : une image imite la présentation d’un média connu ;
- **contenu satirique repris comme une information réelle**.

Une opinion controversée reste un jugement subjectif. Une désinformation porte sur des faits vérifiables et cherche à tromper. 
L’ETL ne doit donc pas transformer automatiquement une opinion, une satire ou un contenu impopulaire en label `fake`.

## 4. Comparaison des sources identifiées

| Source | Modalités disponibles | Format et accès | Langue | Labels et qualité | Méthode d’extraction proposée |
|---|---|---|---|---|---|
| [Fakeddit](https://github.com/entitize/Fakeddit) | Titre de post, image, métadonnées Reddit, commentaires selon les fichiers | TSV et fichiers d’images à télécharger | Anglais | Labels à 2, 3 ou 6 classes. **Qualité moyenne** : annotation à grande échelle par supervision distante, donc bruit possible | Téléchargement officiel, puis `pandas` et `Requests` pour les images manquantes |
| [FakeNewsNet](https://github.com/KaiDMML/FakeNewsNet) | Texte d’article, images, date, éditeur et données sociales selon disponibilité | Index CSV, données JSON et scripts officiels | Anglais | `fake`/`real` issus de PolitiFact et GossipCop. **Qualité moyenne à élevée**, mais variable selon la collection | Scripts officiels du dépôt ; lecture CSV/JSON ; téléchargement contrôlé avec `Requests` |
| [NewsCLIPpings](https://github.com/g-luo/news_clippings) | Image d’actualité et légende associée ou falsifiée | Métadonnées JSON et images de VisualNews | Anglais | Couple image-texte `pristine` ou `falsified`. **Qualité élevée pour l’incohérence multimodale**, mais ce n’est pas un verdict factuel complet | Téléchargement officiel et lecture JSON ; pas de scraping |
| [COSMOS](https://github.com/shivangi-aneja/COSMOS) | Image, légendes provenant de contextes différents et URL d’article | Fichiers d’annotations et images accessibles selon les instructions du projet | Principalement anglais | Annotation d’utilisation hors contexte. **Qualité élevée** pour ce cas précis, mais couverture plus spécialisée | Téléchargement officiel et script Python fourni par le projet |
| [MuMiN](https://mumin-dataset.github.io/) | Claims, tweets/posts, images, articles, utilisateurs et relations | Tables et graphe via package Python ; certaines données doivent être réhydratées | 41 langues annoncées, dont le français | Labels dérivés de sites de fact-checking. **Qualité moyenne à élevée**, avec bruit possible lors de la mise en relation automatique | Package officiel ; APIs des plateformes si les conditions d’accès le permettent |
| [NewsData.io](https://newsdata.io/documentation) | Titre, résumé, contenu, image, date, auteur, pays, catégorie et source | API REST, réponse JSON | Multilingue, dont français | Aucun label vrai/faux. **Label absent** | `Requests` sur l’API officielle, avec pagination et téléchargement séparé des images |
| [Google Fact Check Tools API](https://developers.google.com/fact-check/tools/api/reference/rest/v1alpha1/claims/search) | Claim, auteur du claim, verdict textuel, fact-checker, date et URL ; image non garantie | API REST, réponse JSON | Multilingue | Verdict produit par un fact-checker. **Qualité élevée et traçable**, mais les échelles de verdict sont hétérogènes | `Requests` sur l’API officielle ; enrichissement éventuel depuis la page source |
| [NewsAPI](https://newsapi.org/docs) | Titre, description, extrait, URL d’image, date, auteur et source | API REST, réponse JSON | Plusieurs langues, dont français | Aucun label vrai/faux. **Label absent** | `Requests` ; solution de remplacement à NewsData.io |
| [PolitiFact](https://www.politifact.com/rss/) | Claim, verdict, résumé, date, URL d’article et illustrations de page | Flux RSS/XML et pages HTML | Principalement anglais | Échelle Truth-O-Meter. **Qualité élevée**, avec justification éditoriale | `Feedparser` pour le RSS, puis `Requests` + `Beautiful Soup` pour la page et son image |
| [The Conversation France](https://theconversation.com/fr) | Titre, article complet, auteur, date, image principale et légende | Flux Atom puis pages HTML publiques | Français | Aucun label vrai/faux. **Label absent** : source éditoriale de référence, à ne pas convertir automatiquement en `true` | `Feedparser` pour découvrir les URLs, puis `Requests` + `Beautiful Soup` sur un petit nombre d’articles |
| [Reddit Data API](https://redditinc.com/policies/data-api-terms) | Titre, texte, image ou lien, commentaires, auteur et date | API REST JSON, OAuth ; bibliothèque PRAW possible | Multilingue | Pas de label fiable natif. Flairs et votes : **qualité faible** | API officielle/PRAW ; source secondaire, non retenue pour le pipeline principal |


## 5. Analyse des sources les plus utiles

### 5.1 Fakeddit — dataset multimodal volumineux

Fakeddit est directement adapté à l’étude de contenus trompeurs sur les réseaux sociaux. Chaque exemple peut associer un titre Reddit à une image. Le dataset propose plusieurs niveaux de classification, ce qui le rend pratique pour comprendre qu’un label ne se limite pas toujours à `true` ou `fake`.

**Avantages :** volume important, texte et image déjà associés, format tabulaire simple.  
**Limites :** corpus anglophone ; labels issus d’une supervision distante et donc potentiellement bruités.

**Usage possible dans ce projet :** source batch permettant de tester l’import de fichiers TSV et d’images, sans entraîner de modèle.

### 5.2 FakeNewsNet — articles et fact-checking

FakeNewsNet regroupe deux collections : PolitiFact pour les sujets politiques et GossipCop pour les actualités people. Les fichiers contiennent des identifiants et des métadonnées ; les scripts du dépôt permettent de récupérer les contenus encore disponibles.

**Avantages :** labels documentés, données proches d’un cas réel d’article de presse, métadonnées sociales possibles.  
**Limites :** dépendance à des pages et APIs externes ; liens anciens parfois indisponibles ; qualité différente entre PolitiFact et GossipCop.

**Usage possible dans ce projet :** source batch de référence pour étudier l’import CSV/JSON et la gestion des URLs devenues invalides.

### 5.3 NewsCLIPpings et COSMOS — cohérence entre texte et image

Ces deux datasets portent spécifiquement sur la relation entre une image et son contexte textuel. Ils sont utiles pour les cas où l’image est réelle, mais associée à une mauvaise légende.

Leurs labels ne signifient pas nécessairement que tout l’article est faux. Ils indiquent surtout si le **couple image-texte** est cohérent ou hors contexte. Cette distinction doit être conservée dans le schéma de données.

**Usage possible dans ce projet :** exemples de formats d’annotation spécialisés ; import facultatif si le temps le permet.

### 5.4 NewsData.io — meilleure source pour l’ETL automatisé

NewsData.io fournit des actualités récentes sous forme de JSON. Les réponses peuvent inclure le titre, la description, le contenu, l’URL de l’article, l’URL d’image, la date, la langue et la source.

**Avantages :** API officielle, appels simples, données récentes, filtre de langue, aucune navigation dans un navigateur.  
**Limites :** contenus parfois tronqués, image parfois absente, quota du plan gratuit, aucun label vrai/faux.

**Usage possible dans ce projet :** source principale pour démontrer une extraction automatisée avec `Requests`, pagination, contrôles de champs et téléchargement d’images.

### 5.5 Google Fact Check Tools API — labels traçables

Cette API permet de rechercher des claims déjà examinés par des organisations de fact-checking. Elle renvoie notamment le texte de l’affirmation, son auteur, la date, le nom du fact-checker, l’URL de la vérification et un verdict textuel.

**Avantages :** accès officiel, provenance claire du verdict, plusieurs langues.  
**Limites :** l’image n’est pas garantie ; les verdicts ne suivent pas tous la même échelle (`False`, `Mostly false`, etc.).

**Usage possible dans ce projet :** source complémentaire de métadonnées de fact-checking. Le verdict est conservé tel quel dans `source_label_raw`, sans le convertir automatiquement en vrai/faux.

### 5.6 RSS et pages HTML — PolitiFact ou autre média autorisé

Un flux RSS fournit une liste structurée de publications récentes. Il contient souvent le titre, la date, le résumé et le lien de l’article. Si l’image n’est pas présente dans le flux, la page HTML peut être consultée pour lire sa balise Open Graph `og:image`.

**Avantages :** format léger, souvent sans authentification, adapté à une collecte périodique.  
**Limites :** contenu parfois incomplet ; structure HTML susceptible de changer ; il faut respecter les règles du site.

**Usage possible dans ce projet :** démonstration pédagogique de `Feedparser`, puis `Requests` et `Beautiful Soup` sur un petit nombre de pages.

### 5.7 The Conversation France — extraction HTML pédagogique

Le flux Atom fournit une liste d’articles récents. Pour chaque URL, la page HTML contient un titre `h1`, un corps identifié par `itemprop="articleBody"`, une date dans une balise `time`, un auteur et une image principale via `og:image`.

**Avantages :** articles français en accès libre, HTML sémantique, texte et image clairement associés, absence de clé API. Le flux annonce une licence Creative Commons avec attribution et sans modification.

**Limites :** aucun label vrai/faux ; les sélecteurs HTML peuvent changer ; les droits propres aux photographies doivent rester associés à leur légende et à la page source.

**Usage possible dans ce projet :** quatrième source limitée à quelques pages, avec une pause entre les articles. Le script conserve l’URL, l’auteur, la mention de licence et la légende de l’image. Il ne contourne ni connexion, ni paywall, ni CAPTCHA.

## 6. Outils d’extraction et cas d’utilisation

| Outil | Rôle | Source adaptée dans le projet | Quand l’utiliser |
|---|---|---|---|
| `Requests` | Envoyer des requêtes HTTP et télécharger JSON, HTML ou images | NewsData.io, Google Fact Check API, NewsAPI | Une API ou une URL directe est disponible |
| `Feedparser` | Lire et convertir un flux RSS/Atom | RSS de PolitiFact ou d’un média | Le site publie un flux structuré |
| `Beautiful Soup` | Extraire quelques champs d’une page HTML | Articles The Conversation obtenus depuis le flux Atom | Le HTML est déjà reçu avec `Requests` et peu de pages sont parcourues |
| `Scrapy` | Explorer beaucoup de pages avec files d’attente, règles et pipelines | Archives publiques d’un site de fact-checking | Plusieurs dizaines ou centaines de pages doivent être parcourues régulièrement |
| `Selenium` | Piloter un navigateur et exécuter JavaScript | Démonstration sur une page dynamique sans API équivalente | Le contenu apparaît seulement après interaction ou exécution JavaScript |
| `pandas` | Lire, filtrer et convertir CSV/TSV/JSON | Fakeddit et FakeNewsNet | Un dataset est déjà disponible sous forme de fichiers |


## 7. Format de sortie proposé

### 7.1 Organisation des fichiers

```text
data/
├── raw/
│   ├── newsdata/2026-08-26.jsonl
│   ├── rss/2026-08-26.xml
│   ├── theconversation/2026-09-02.json
│   └── datasets/
├── images/
│   └── <publication_id>.jpg
├── processed/
│   └── publications.parquet
└── rejected/
    └── invalid_records.jsonl
```

- **JSONL brut** : conserve la réponse d’origine, une publication par ligne.
- **Parquet normalisé** : compact, typé et efficace pour l’analyse.
- **Dossier images** : évite de dépendre uniquement d’URLs susceptibles de disparaître.
- **Rejets JSONL** : conserve les entrées incomplètes avec la raison du rejet.

CSV reste possible pour une vérification manuelle, mais il gère moins bien les textes longs, les listes et les champs imbriqués.

### 7.2 Schéma commun d’une publication

| Champ | Règle |
|---|---|
| `publication_id` | Identifiant stable et unique, préfixé par la source |
| `source_name` | Nom de la source d’acquisition |
| `source_domain` | Domaine de la publication, sans `www.` |
| `source_url` | URL de la publication, pas celle de l’API |
| `title` | Titre de la publication |
| `text` | Contenu, résumé ou affirmation examinée |
| `image_url` | URL de l’image associée à cette publication |
| `image_path` | Chemin local après téléchargement réussi |
| `published_at` | Date fournie par la source, normalisée en UTC |
| `language` | Code ISO comme `fr` ou `en` |
| `author` | Chaîne ou `null` |
| `source_label_raw` | Label original, sans modification |
| `source_label_scheme` | Système de labels utilisé |
| `label_provenance` | Organisation ou méthode ayant produit le label |
| `collected_at` | Date UTC de l’extraction |

Règles importantes
- source_url désigne l’article ou le post, jamais https://newsdata.io/api/....
- image_path reste null jusqu’au téléchargement réussi.
- published_at et collected_at ne doivent pas être confondus.
- Les champs de labels existent toujours, même lorsqu’ils valent null.
- Un extracteur ne doit jamais inventer un label manquant.
- Utilise null dans JSON, mais None dans Python.

```json
{
  "publication_id": "newsdata_12345",
  "source_name": "NewsData.io",
  "source_domain": "example.org",
  "source_url": "https://example.org/article",
  "title": "Titre de la publication",
  "text": "Texte ou résumé disponible",
  "image_url": "https://example.org/image.jpg",
  "image_path": "data/images/newsdata_12345.jpg",
  "published_at": "2026-08-26T08:00:00Z",
  "language": "fr",
  "author": null,
  "source_label_raw": null,
  "source_label_scheme": null,
  "label_provenance": null,
  "collected_at": "2026-08-26T09:00:00Z"
}
```


Pour un dataset labellisé, `source_label_raw` conserve la valeur originale, par exemple `mostly-false` ou `out-of-context`. Le pipeline ne la transforme pas silencieusement en `fake`.

## 8. Workflow ETL proposé

```text
Sources
  ↓
Extracteurs dédiés (API, RSS, HTML, fichiers)
  ↓
Données brutes JSONL/XML + images
  ↓
Normalisation vers le schéma commun
  ↓
Contrôles qualité et dédoublonnage
  ↓
Parquet final + journal des rejets
```

### Extract

- appeler les APIs avec `Requests` ;
- parcourir le RSS avec `Feedparser` ;
- compléter quelques pages avec `Beautiful Soup` ;
- importer les CSV, TSV et JSON des datasets ;
- télécharger l’image et mémoriser son URL d’origine.

### Transform

- uniformiser les dates en UTC et la langue en code ISO ;
- nettoyer les espaces sans altérer le texte ;
- générer un identifiant stable à partir de la source et de l’URL ;
- conserver les labels et leur provenance dans des champs séparés ;
- calculer un hash de l’URL ou de l’image pour détecter les doublons.

### Load

- écrire les données brutes sans les écraser ;
- enregistrer les images valides ;
- produire le fichier Parquet normalisé ;
- isoler les publications invalides dans le journal des rejets.

### Contrôles de qualité minimaux

Une publication est acceptée si :

- son titre ou son texte n’est pas vide ;
- son URL de publication est valide ;
- son image est rattachée à la même entrée et a pu être téléchargée ;
- son type MIME commence par `image/` ;
- sa date et sa source sont conservées ;
- le même identifiant n’existe pas déjà.

Le workflow doit être relançable sans créer de doublons. Il doit utiliser des timeouts, quelques nouvelles tentatives sur les erreurs temporaires, un journal d’exécution et un point de reprise pour la pagination.

## 9. Sources retenues pour le projet

Pour rester simple tout en montrant plusieurs méthodes d’acquisition, nous suivrons la combinaison suivante :

1. **Source principale : NewsData.io avec Requests**  
   Elle fournit simplement des actualités récentes, multilingues, avec texte, métadonnées et souvent une image. Elle convient au pipeline automatisé principal.

2. **Source complémentaire : flux RSS de PolitiFact avec Feedparser**  
   Il permet de manipuler des données structurées différentes d’une API et d’observer des verdicts de fact-checking.

3. **Source HTML directe : The Conversation France avec Beautiful Soup**

   Le flux Atom fournit les URLs, puis Beautiful Soup extrait le titre, le corps, l’auteur, la date et l’image Open Graph sur un nombre limité de pages.

4. **Jeux de données de référence : Fakeddit**  
   Ils montrent différents formats et différentes définitions des labels. Ils peuvent être importés en batch pour valider le schéma ETL, sans entraîner de modèle dans le périmètre actuel.

La sortie recommandée est **JSONL pour les données brutes**, **Parquet pour les publications normalisées** et un **dossier local pour les images**. Cette solution reste claire, modulaire, relançable sans intervention et adaptée à cette phase du projet.


## Références principales

- [Multimodal Fake News Detection: A Survey](https://www.ijci.zu.edu.eg/index.php/ijci/article/view/102/86)
- [Fakeddit — dépôt officiel](https://github.com/entitize/Fakeddit)
- [FakeNewsNet — dépôt officiel](https://github.com/KaiDMML/FakeNewsNet)
- [NewsCLIPpings — dépôt officiel](https://github.com/g-luo/news_clippings)
- [COSMOS — dépôt officiel](https://github.com/shivangi-aneja/COSMOS)
- [MuMiN — site du dataset](https://mumin-dataset.github.io/)
- [NewsData.io — documentation](https://newsdata.io/documentation)
- [Google Fact Check Tools API — documentation](https://developers.google.com/fact-check/tools/api/reference/rest/v1alpha1/claims/search)
- [NewsAPI — documentation](https://newsapi.org/docs)
- [The Conversation France — flux Atom](https://theconversation.com/fr/articles.atom)
- [The Conversation — robots.txt](https://theconversation.com/robots.txt)
- [The Conversation — charte éditoriale et republication](https://cdn.theconversation.com/static_files/files/1878/TC_Global_Editorial_Guidelines_FA_TC-FR.pdf)
- [CNIL — collecte par moissonnage web](https://www.cnil.fr/fr/focus-interet-legitime-collecte-par-moissonnage-web-scraping)
